#  House Price Prediction — Interactive Notebook

This notebook walks through the entire ML pipeline step by step.
It is designed for students, data science learners, and GitHub portfolio projects.

---
**Author:** Your Name  
**Dataset:** Synthetic Housing Data (2,000 samples)  
**Models:** Linear Regression | Decision Tree | Random Forest | XGBoost  

---

##  Import Libraries

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data_generator import generate_housing_data
from preprocessor   import clean_data, feature_engineering, prepare_features
from models         import train_all_models, save_best_model
from predictor      import build_input_vector, predict_price, print_prediction_report

%matplotlib inline
print(' All imports successful!')

##  Generate & Load Dataset

In [ ]:
df = generate_housing_data(n_samples=2000)
print(f'Dataset shape: {df.shape}')
df.head(10)

In [ ]:
df.describe().round(2)

In [ ]:
# Check for missing values
print('Missing values:')
print(df.isnull().sum())

##  Exploratory Data Analysis (EDA)

In [ ]:
# Price Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['price_usd'], bins=40, kde=True, color='#2196F3', ax=axes[0])
axes[0].set_title('House Price Distribution', fontweight='bold')
axes[0].set_xlabel('Price (USD)')

sns.histplot(np.log1p(df['price_usd']), bins=40, kde=True, color='#FF5722', ax=axes[1])
axes[1].set_title('Log-Transformed Price Distribution', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(12, 9))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, annot_kws={'size': 8})
plt.title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Area vs Price
plt.figure(figsize=(10, 6))
plt.scatter(df['area_sqft'], df['price_usd'], alpha=0.3, s=15, color='#4C72B0')
m, b = np.polyfit(df['area_sqft'], df['price_usd'], 1)
x_line = np.linspace(df['area_sqft'].min(), df['area_sqft'].max(), 100)
plt.plot(x_line, m*x_line + b, 'r--', linewidth=2, label='Trend line')
plt.xlabel('Area (sq ft)'); plt.ylabel('Price (USD)')
plt.title('Area vs House Price', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

##  Data Cleaning & Feature Engineering

In [ ]:
df_clean = clean_data(df.copy())
df_eng   = feature_engineering(df_clean.copy())
print('Columns after engineering:')
print(df_eng.columns.tolist())
df_eng.head()

##  Train-Test Split + Scaling

In [ ]:
X_train, X_test, y_train, y_test, scaler, feature_names = prepare_features(df_eng)
print(f'Training set: {X_train.shape}')
print(f'Testing  set: {X_test.shape}')
print(f'Features    : {feature_names}')

##  Train & Evaluate All Models

In [ ]:
results = train_all_models(X_train, y_train, X_test, y_test)

##  Actual vs Predicted Plot

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

for ax, (name, (model, y_pred, metrics)), color in zip(axes, results.items(), colors):
    ax.scatter(y_test, y_pred, alpha=0.4, s=15, color=color)
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect fit')
    ax.set_title(f'{name}  |  R²={metrics["R2"]:.4f}', fontweight='bold')
    ax.set_xlabel('Actual Price'); ax.set_ylabel('Predicted Price')
    ax.legend()

plt.suptitle('Actual vs Predicted House Prices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

##  Predict Price for a New House

In [ ]:
# --- Change these values to predict for any house ---
my_house = {
    'area_sqft'     : 1800,      # sq ft
    'bedrooms'      : 3,
    'bathrooms'     : 2,
    'floors'        : 1,
    'age_years'     : 5,
    'garage'        : 1,
    'garden'        : 0,
    'swimming_pool' : 0,
    'location_score': 6.0,
    'distance_city' : 10.0,
    'school_nearby' : 1,
    'furnishing'    : 1,          # 0=unfurnished, 1=semi, 2=fully
}

# Use best model (Linear Regression in our case)
best_name  = max(results, key=lambda k: results[k][2]['R2'])
best_model = results[best_name][0]

vector    = build_input_vector(my_house, scaler, feature_names)
predicted = predict_price(best_model, vector)
print_prediction_report(my_house, predicted, best_name)

---
##  Project Complete!

**What we did:**
1. Generated 2,000 realistic synthetic housing records
2. Cleaned data (removed duplicates & outliers)
3. Engineered 4 new features (total_rooms, amenity_score, age_category, location_tier)
4. Split into 80/20 train-test and applied StandardScaler
5. Trained 4 regression models
6. Evaluated using MAE, RMSE, R²
7. Predicted price for a new custom property

**Upload this to GitHub as proof of work!** 🚀